In [ ]:
!apt-get update -qq
!apt-get install -y poppler-utils -qq

!pip install pdf2image -q

print("✅ PDF 변환 환경 준비 완료")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
from pdf2image import convert_from_path

os.makedirs("/content/document_images", exist_ok=True)

pdf_files = [
    "lease_A.pdf",
    "payslip_A.pdf",
    "bill_A.pdf"
]

for pdf_file in pdf_files:
    pages = convert_from_path(
        pdf_file,
        dpi=300
    )

    base_name = os.path.splitext(pdf_file)[0]

    for page_number, page in enumerate(pages, start=1):
        output_path = (
            f"/content/document_images/"
            f"{base_name}_page_{page_number}.png"
        )

        page.save(
            output_path,
            "PNG"
        )

        print("저장 완료:", output_path)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

image_files = sorted(
    os.listdir("/content/document_images")
)

plt.figure(figsize=(15, 7))

for i, image_file in enumerate(image_files):
    image_path = os.path.join(
        "/content/document_images",
        image_file
    )

    image = Image.open(image_path)

    plt.subplot(1, len(image_files), i + 1)
    plt.imshow(image)
    plt.title(image_file)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np

os.makedirs(
    "/content/document_images_degraded",
    exist_ok=True
)

def make_camera_version(input_path, output_path):
    image = cv2.imread(input_path)

    height, width = image.shape[:2]

    # 약간 회전
    rotation_matrix = cv2.getRotationMatrix2D(
        (width / 2, height / 2),
        2.2,
        0.92
    )

    rotated = cv2.warpAffine(
        image,
        rotation_matrix,
        (width, height),
        borderValue=(235, 235, 235)
    )

    # 밝기와 대비를 조금 낮춤
    degraded = cv2.convertScaleAbs(
        rotated,
        alpha=0.85,
        beta=15
    )

    # 미세한 흐림 추가
    degraded = cv2.GaussianBlur(
        degraded,
        (3, 3),
        0
    )

    cv2.imwrite(
        output_path,
        degraded
    )


for image_file in image_files:
    input_path = os.path.join(
        "/content/document_images",
        image_file
    )

    output_file = image_file.replace(
        ".png",
        "_camera.png"
    )

    output_path = os.path.join(
        "/content/document_images_degraded",
        output_file
    )

    make_camera_version(
        input_path,
        output_path
    )

    print("촬영형 이미지 저장:", output_path)

In [ ]:
original_path = (
    "/content/document_images/"
    "lease_A_page_1.png"
)

camera_path = (
    "/content/document_images_degraded/"
    "lease_A_page_1_camera.png"
)

original = Image.open(original_path)
camera = Image.open(camera_path)

plt.figure(figsize=(13, 7))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original Document")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(camera)
plt.title("Camera-like Document")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import shutil

shutil.make_archive(
    "/content/DAEGUIDE_OCR_images",
    "zip",
    "/content/document_images"
)

shutil.make_archive(
    "/content/DAEGUIDE_OCR_camera_images",
    "zip",
    "/content/document_images_degraded"
)

In [ ]:
files.download(
    "/content/DAEGUIDE_OCR_images.zip"
)

In [ ]:
files.download(
    "/content/DAEGUIDE_OCR_camera_images.zip"
)

In [ ]:
!pip install -q easyocr

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import easyocr
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

reader = easyocr.Reader(
    ['ko', 'en'],
    gpu=True
)

image_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(('.png', '.jpg', '.jpeg'))
]

print("업로드된 이미지:", image_files)

In [ ]:
ocr_results = {}

for image_path in image_files:
    results = reader.readtext(
        image_path,
        detail=1,
        paragraph=False
    )

    ocr_results[image_path] = results

    print("\n" + "=" * 60)
    print("파일:", image_path)
    print("=" * 60)

    for box, text, confidence in results:
        print(f"{text} / 신뢰도: {confidence:.3f}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

for image_path, results in ocr_results.items():
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    for box, text, confidence in results:
        points = [[int(x), int(y)] for x, y in box]
        x_values = [p[0] for p in points]
        y_values = [p[1] for p in points]

        x1, x2 = min(x_values), max(x_values)
        y1, y2 = min(y_values), max(y_values)

        color = (31, 120, 92) if confidence >= 0.7 else (255, 120, 80)

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            color,
            3
        )

    plt.figure(figsize=(12, 16))
    plt.imshow(image)
    plt.title(f"OCR result: {image_path}")
    plt.axis("off")
    plt.show()

In [ ]:
rows = []

for image_path, results in ocr_results.items():
    for box, text, confidence in results:
        rows.append({
            "file_name": image_path,
            "recognized_text": text,
            "confidence": round(float(confidence), 4)
        })

ocr_df = pd.DataFrame(rows)

display(ocr_df.head(20))
ocr_df.to_csv(
    "document_ocr_results.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
files.download("document_ocr_results.csv")

In [ ]:
document_texts = {}

for image_path, results in ocr_results.items():
    full_text = "\n".join([
        text for box, text, confidence in results
        if confidence >= 0.3
    ])

    document_texts[image_path] = full_text

    print("\n" + "=" * 60)
    print(image_path)
    print("=" * 60)
    print(full_text)

In [ ]:
!pip install -q "google-auth==2.49.0" "google-genai" "pydantic" --upgrade-strategy only-if-needed

In [ ]:
from google import genai
from pydantic import BaseModel
from getpass import getpass

print("라이브러리 불러오기 성공")

In [ ]:
!pip install -q -U google-genai pydantic

In [ ]:
import os
from getpass import getpass
from google import genai

api_key = getpass("Gemini API Key를 입력하세요: ")

client = genai.Client(api_key=api_key)

print("Gemini 연결 준비 완료")

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal


class MoneyItem(BaseModel):
    name: str = Field(description="금액의 의미. 예: 보증금, 월세, 실수령액")
    amount: str = Field(description="문서에 적힌 금액")
    evidence: str = Field(description="판단 근거가 된 원문")


class DateItem(BaseModel):
    name: str = Field(description="날짜의 의미. 예: 납부기한, 계약 종료일")
    date: str = Field(description="문서에 적힌 날짜")
    evidence: str = Field(description="판단 근거가 된 원문")


class RiskItem(BaseModel):
    level: Literal["낮음", "확인 필요", "주의"]
    title: str = Field(description="주의할 조건의 짧은 제목")
    explanation: str = Field(description="외국인도 이해하기 쉬운 설명")
    evidence: str = Field(description="판단 근거가 된 원문")


class DocumentAnalysis(BaseModel):
    document_type: Literal[
        "임대차계약서",
        "급여명세서",
        "공과금고지서",
        "기타"
    ]
    summary: str = Field(description="문서의 핵심 내용을 쉬운 한국어로 요약")
    money_items: List[MoneyItem]
    date_items: List[DateItem]
    risks: List[RiskItem]
    next_actions: List[str] = Field(description="사용자가 다음으로 해야 할 행동")
    missing_or_unclear: List[str] = Field(
        description="OCR 오류 또는 문서에서 확인할 수 없는 내용"
    )

In [ ]:
def analyze_financial_document(ocr_text):
    prompt = f"""
당신은 한국에서 생활하는 외국인을 위한 금융 문서 분석 도우미입니다.

아래 내용은 OCR로 추출한 문서이므로 글자나 띄어쓰기가 틀릴 수 있습니다.

분석 원칙:
1. 문서에 실제로 적힌 정보만 사용하세요.
2. 확인되지 않은 금액이나 날짜를 추측하지 마세요.
3. 보증금, 월세, 실수령액, 공제액, 납부금액을 찾아주세요.
4. 계약기간, 지급일, 납부기한을 찾아주세요.
5. 연체료, 위약금, 중도해지, 자동갱신, 추가 비용 등 불리할 수 있는 조건을 설명하세요.
6. 어려운 법률·금융 용어는 쉬운 한국어로 설명하세요.
7. OCR 때문에 불분명한 내용은 missing_or_unclear에 기록하세요.
8. 법률적 결론을 단정하지 말고 확인이 필요한 사항으로 표현하세요.
9. 각 금액·날짜·위험 항목에는 반드시 근거 원문을 포함하세요.

OCR 추출 내용:
--------------------
{ocr_text}
--------------------
"""

    interaction = client.interactions.create(
        model="gemini-3.5-flash",
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": DocumentAnalysis.model_json_schema()
        }
    )

    return DocumentAnalysis.model_validate_json(
        interaction.output_text
    )

In [ ]:
print(document_texts.keys())

In [ ]:
from pydantic import BaseModel
from typing import List, Literal


class MoneyItem(BaseModel):
    name: str
    amount: str
    evidence: str


class DateItem(BaseModel):
    name: str
    date: str
    evidence: str


class RiskItem(BaseModel):
    level: Literal["낮음", "확인 필요", "주의"]
    title: str
    explanation: str
    evidence: str


class DocumentAnalysis(BaseModel):
    document_type: Literal[
        "임대차계약서",
        "급여명세서",
        "공과금고지서",
        "기타"
    ]
    summary: str
    money_items: List[MoneyItem]
    date_items: List[DateItem]
    risks: List[RiskItem]
    next_actions: List[str]
    missing_or_unclear: List[str]


def analyze_financial_document(ocr_text):
    prompt = f"""
당신은 한국에서 생활하는 외국인을 위한 금융 문서 분석 도우미입니다.

아래 OCR 문서를 분석하세요.

규칙:
1. 문서에 실제로 적힌 정보만 사용하세요.
2. 금액과 날짜를 추측하지 마세요.
3. 보증금, 월세, 실수령액, 공제액, 납부액을 찾으세요.
4. 계약기간, 지급일, 납부기한을 찾으세요.
5. 연체료, 위약금, 중도해지, 자동갱신 등 위험 조건을 찾으세요.
6. 어려운 금융·계약 내용을 쉬운 한국어로 설명하세요.
7. OCR 때문에 불분명한 내용은 missing_or_unclear에 기록하세요.
8. 금액·날짜·위험 항목에는 근거 원문을 포함하세요.
9. 확인되지 않은 내용을 만들어내지 마세요.

OCR 문서:
----------------
{ocr_text}
----------------
"""

    interaction = client.interactions.create(
        model="gemini-3.5-flash",
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": DocumentAnalysis.model_json_schema()
        }
    )

    return DocumentAnalysis.model_validate_json(
        interaction.output_text
    )


print("✅ AI 문서 분석 함수 생성 완료")

In [ ]:
file_name = "lease_A_page_1_camera.png"
ocr_text = document_texts[file_name]

result = analyze_financial_document(ocr_text)

print(result.model_dump_json(indent=2))

In [ ]:
print("📄 문서 종류")
print(result.document_type)

print("\n📝 쉬운 설명")
print(result.summary)

print("\n💰 주요 금액")
for item in result.money_items:
    print(f"• {item.name}: {item.amount}")
    print(f"  근거: {item.evidence}")

print("\n📅 주요 일정")
for item in result.date_items:
    print(f"• {item.name}: {item.date}")
    print(f"  근거: {item.evidence}")

print("\n⚠ 확인이 필요한 내용")
for risk in result.risks:
    print(f"• [{risk.level}] {risk.title}")
    print(f"  {risk.explanation}")
    print(f"  근거: {risk.evidence}")

print("\n✅ 다음 행동")
for action in result.next_actions:
    print("•", action)

print("\n❓ 정확히 읽히지 않은 내용")
for item in result.missing_or_unclear:
    print("•", item)

In [ ]:
all_ai_results = {}

for file_name, ocr_text in document_texts.items():
    try:
        print(f"분석 중: {file_name}")

        analysis = analyze_financial_document(ocr_text)
        all_ai_results[file_name] = analysis.model_dump()

        print("완료")

    except Exception as e:
        print("분석 실패:", e)

In [ ]:
import json

with open(
    "daeguide_document_analysis.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        all_ai_results,
        file,
        ensure_ascii=False,
        indent=2
    )

print("JSON 저장 완료")

In [ ]:
from google.colab import files

files.download("daeguide_document_analysis.json")

In [ ]:
from pydantic import BaseModel
from typing import List


class TranslatedAnalysis(BaseModel):
    language: str
    summary: str
    risks: List[str]
    next_actions: List[str]

In [ ]:
def translate_analysis(analysis_data, target_language):
    language_names = {
        "en": "English",
        "zh": "Simplified Chinese",
        "vi": "Vietnamese"
    }

    target_name = language_names[target_language]

    prompt = f"""
다음 금융 문서 분석 결과를 {target_name}로 번역하세요.

번역 원칙:
1. 금액과 날짜는 원본 그대로 유지하세요.
2. 외국인이 이해하기 쉬운 표현을 사용하세요.
3. 어려운 금융·법률 용어는 쉬운 말로 바꾸세요.
4. 새로운 정보를 추가하거나 추측하지 마세요.
5. 위험 안내는 지나치게 단정하지 말고 확인이 필요하다고 표현하세요.

분석 결과:
{analysis_data}
"""

    models = [
        "gemini-3.5-flash",
        "gemini-3.1-flash-lite",
        "gemini-2.5-flash"
    ]

    for model_name in models:
        try:
            interaction = client.interactions.create(
                model=model_name,
                input=prompt,
                response_format={
                    "type": "text",
                    "mime_type": "application/json",
                    "schema": TranslatedAnalysis.model_json_schema()
                }
            )

            return TranslatedAnalysis.model_validate_json(
                interaction.output_text
            )

        except Exception as e:
            print(f"{model_name} 실패, 다음 모델 시도")
            last_error = e

    raise RuntimeError(last_error)

In [ ]:
import json

analysis_dict = result.model_dump()

analysis_text = json.dumps(
    analysis_dict,
    ensure_ascii=False,
    indent=2
)

english_result = translate_analysis(analysis_text, "en")
chinese_result = translate_analysis(analysis_text, "zh")
vietnamese_result = translate_analysis(analysis_text, "vi")

In [ ]:
print("🇺🇸 영어")
print(english_result.model_dump_json(indent=2))

print("\n🇨🇳 중국어")
print(chinese_result.model_dump_json(indent=2))

print("\n🇻🇳 베트남어")
print(vietnamese_result.model_dump_json(indent=2))

In [ ]:
multilingual_result = {
    "file_name": "lease_A_page_1_camera.png",
    "original_analysis": result.model_dump(),
    "translations": {
        "en": english_result.model_dump(),
        "zh": chinese_result.model_dump(),
        "vi": vietnamese_result.model_dump()
    }
}

with open(
    "daeguide_multilingual_analysis.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        multilingual_result,
        file,
        ensure_ascii=False,
        indent=2
    )

print("✅ 다국어 분석 파일 저장 완료")

In [ ]:
from google.colab import files

files.download("daeguide_multilingual_analysis.json")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import json

with open(
    "daeguide_document_analysis.json",
    "r",
    encoding="utf-8"
) as file:
    all_ai_results = json.load(file)

print("문서 개수:", len(all_ai_results))
print("파일 목록:", list(all_ai_results.keys()))

In [ ]:
from google import genai
from getpass import getpass

api_key = getpass("Gemini API Key를 입력하세요: ")
client = genai.Client(api_key=api_key)

print("✅ Gemini 연결 완료")

In [ ]:
import json
import time
from pydantic import BaseModel
from typing import List


class TranslatedAnalysis(BaseModel):
    language: str
    summary: str
    risks: List[str]
    next_actions: List[str]


def translate_analysis(analysis_data, target_language):
    language_names = {
        "en": "English",
        "zh": "Simplified Chinese",
        "vi": "Vietnamese"
    }

    prompt = f"""
다음 금융 문서 분석 결과를 {language_names[target_language]}로 번역하세요.

규칙:
1. 금액과 날짜를 원문 그대로 유지하세요.
2. 어려운 금융·계약 용어를 쉬운 말로 설명하세요.
3. 새로운 정보를 추가하거나 추측하지 마세요.
4. 위험 내용은 단정하지 말고 확인이 필요하다고 표현하세요.
5. summary에는 문서 전체 요약을 작성하세요.
6. risks에는 주의할 사항을 작성하세요.
7. next_actions에는 사용자가 다음으로 해야 할 행동을 작성하세요.

분석 결과:
{json.dumps(analysis_data, ensure_ascii=False, indent=2)}
"""

    model_names = [
        "gemini-3.5-flash",
        "gemini-3.1-flash-lite",
        "gemini-2.5-flash"
    ]

    last_error = None

    for model_name in model_names:
        try:
            interaction = client.interactions.create(
                model=model_name,
                input=prompt,
                response_format={
                    "type": "text",
                    "mime_type": "application/json",
                    "schema": TranslatedAnalysis.model_json_schema()
                }
            )

            return TranslatedAnalysis.model_validate_json(
                interaction.output_text
            ).model_dump()

        except Exception as error:
            last_error = error
            print(f"⚠️ {model_name} 실패 → 다음 모델 시도")
            time.sleep(2)

    raise RuntimeError(last_error)

In [ ]:
multilingual_all = {}

languages = ["en", "zh", "vi"]

for file_name, analysis in all_ai_results.items():
    print("\n📄 번역 시작:", file_name)

    multilingual_all[file_name] = {
        "original_analysis": analysis,
        "translations": {}
    }

    for language in languages:
        print(f"  → {language} 번역 중")

        try:
            translated = translate_analysis(
                analysis,
                language
            )

            multilingual_all[file_name]["translations"][language] = translated

            print(f"  ✅ {language} 완료")
            time.sleep(2)

        except Exception as error:
            print(f"  ❌ {language} 실패:", error)

In [ ]:
for file_name, data in multilingual_all.items():
    completed_languages = list(
        data["translations"].keys()
    )

    print(file_name, ":", completed_languages)

In [ ]:
output_file = "daeguide_multilingual_analysis_all.json"

with open(
    output_file,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        multilingual_all,
        file,
        ensure_ascii=False,
        indent=2
    )

print("✅ 저장 완료:", output_file)

In [ ]:
files.download("daeguide_multilingual_analysis_all.json")